In [8]:
import argparse
import os
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [9]:
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torchvision.transforms.functional as RF
from PIL import Image
import numpy as np
import random,cv2,pandas,os,numpy
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [10]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

nz = 100
beta1 = 0.5
lr = 0.0001
batch_size=100000
ngpu=1
ngf,nc = 3,3
ndf = 64

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

Random Seed:  999


In [49]:
x = pandas.read_csv("/kaggle/input/rohlik-sales-forecasting-challenge-v2/sales_train.csv").sort_values(by=['date', 'warehouse'])
x = torch.from_numpy(numpy.nan_to_num(x[['total_orders', 'sell_price_main', 'type_0_discount', 'type_1_discount',
                                         'type_2_discount', 'type_3_discount', 'type_4_discount',  'type_5_discount', 
                                         'type_6_discount']].to_numpy().astype(numpy.float32)))
x

tensor([[4.7970e+03, 5.6900e+01, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [4.7970e+03, 4.5090e+01, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [4.7970e+03, 1.7840e+01, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        ...,
        [5.3920e+03, 1.4490e+01, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [5.1770e+03, 2.7544e+02, 6.9720e-02,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [5.1770e+03, 7.3800e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00]])

In [13]:
class RF_NET(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.rafire = nn.Sequential(       
            nn.Linear(9, 1524),
            nn.Linear(1524, 824),
            nn.Linear(824, 424),
            nn.Linear(424, 124),
            nn.Linear(124, 1)
        )

    def forward(self,x):
        
        return self.rafire(x)


In [14]:
rf_net = RF_NET().type(torch.float32).to(device)
rf_net= nn.DataParallel(rf_net).to(device)

rf_net.load_state_dict(torch.load("/kaggle/input/encoder-weight-data/weight.pth",weights_only=False,map_location=torch.device('cpu')))

<All keys matched successfully>

In [35]:
encode=[]
#j=0
for i in x:
    encode.append(rf_net(i.reshape((1,9))))
    #if j==25:
    #    break
    #j+=1

In [38]:
torch.save(torch.Tensor(encode).reshape((len(encode),1)), 'train_tensor.pt')
#torch.Tensor(encode).reshape((len(encode),1))

tensor([[51.7320],
        [37.1046],
        [46.4465],
        [46.7976],
        [44.7660],
        [53.3733],
        [36.9323],
        [35.4996],
        [41.0548],
        [46.5748],
        [48.4999],
        [29.8003],
        [33.7823],
        [36.6183],
        [41.0376],
        [50.8909],
        [51.8992],
        [42.7974],
        [44.3708],
        [48.3074],
        [55.7038],
        [42.9343],
        [50.5972],
        [44.3238],
        [33.2858],
        [47.8243]])